# HDFS HA ローリングアップグレード

D12で構築した検証用のHDFSを対象に、Standby NameNode → 計画フェイルオーバー → 旧Active NameNode → DataNodeの順で更新します。Java 11、systemd、単一nameservice、NameNode 2台、DataNode 3台以上の構成を想定しています。この版で実行できるのは simple 認証です。

JournalNode、ZooKeeper、YARN、Sparkはこの手順の対象外です。HDFSのデーモンには個別のsystemd drop-inで新バイナリを指定します。共有の `/opt/hadoop/current` は既存クライアントやYARNが参照します。後日の再構築時は、このdrop-inと配布版の管理を揃えてください。

使い捨ての検証クラスタで上から1セルずつ実行してください。更新元・更新先の互換性とリリースノート、バックアップの復旧手順を事前に確認します。フェイルオーバー時はクライアントの再試行による遅延があり得ます。観測した読み書きの失敗件数と時間を記録します。

[Apache Hadoop HDFS Rolling Upgrade](https://hadoop.apache.org/docs/r3.3.6/hadoop-project-dist/hadoop-hdfs/HdfsRollingUpgrade.html)


## 1. 接続先と配布物

SSH接続先はユーザー名付きのホスト名またはSSH configの別名を指定します。各ノードでSSHユーザーのパスワードなしsudoが必要です。NameNodeのキーにはHDFSのHA ID、DataNodeのキーにはRPC接続に使うホスト名を指定します。

3.2.2 → 3.3.6は検証候補です。配布物を変える場合は、対応するURLとApacheのSHA-512を入力してください。


In [ ]:
source_version = "3.2.2"
target_version = "3.3.6"
release_url = "https://archive.apache.org/dist/hadoop/common/hadoop-3.3.6/hadoop-3.3.6.tar.gz"
release_sha512 = "de3eaca2e0517e4b569a88b63c89fae19cb8ac6c01ff990f1ff8f0cc0f3128c8e8a23db01577ca562a0e0bb1b4a3889f8c74384e609cd55e537aada3dcaa9f8a"

nameservice = "hdfs-cluster"
namenodes = {"nn1": "ansible@nn1", "nn2": "ansible@nn2"}
datanodes = {"nn1": "ansible@nn1", "nn2": "ansible@nn2", "dn1": "ansible@dn1"}
client = "ansible@nn1"
ssh_options = []  # 例: ["-F", "/path/to/ssh_config"]


## 2. 接続・HA状態・HDFSの健全性を確認

In [ ]:
from scripts.hdfs_rolling_upgrade import HdfsUpgrade

upgrade = HdfsUpgrade(source_version, target_version, release_url, release_sha512,
                      nameservice, namenodes, datanodes, client, ssh_options)
upgrade.preflight()


## 3. 配布物の展開と読み書きの観測

チェックサムを検証して、各対象ホストに新しい配布物を展開します。バイナリの切り替えは次の段階から行います。観測用のデータは一意なHDFSパスに置き、同じ旧版クライアントで更新前から更新後まで読み書きします。各試行の開始時刻、所要時間、結果をローカルJSONLに記録します。


In [ ]:
upgrade.stage()
upgrade.start_probe()


## 4. ローリングアップグレードを準備

`prepare`を実行し、`query`でロールバック用fsimageの作成完了を待ちます。


In [ ]:
upgrade.prepare()

## 5. Standby NameNodeを更新

現在のActiveとStandbyを取得し、StandbyのZKFCとNameNodeを停止します。新バイナリで `namenode -rollingUpgrade started` とZKFCを起動し、Standbyとして復帰することを確認します。


In [ ]:
upgrade.update_standby()

## 6. フェイルオーバーして旧Activeを更新

`haadmin -failover`で更新済みNameNodeをActiveにします。両側の状態を確認してから旧Activeを更新します。


In [ ]:
upgrade.failover()

In [ ]:
upgrade.update_former_active()

## 7. DataNodeを1台ずつ更新

`shutdownDatanode … upgrade`で停止し、新バイナリで起動します。RPCの復帰とLive DataNode数、欠損ブロックの有無を確認してから次のノードに進みます。


In [ ]:
upgrade.update_datanodes()

## 8. 更新後の確認

旧版クライアントで更新前のデータを読み出して照合し、テストパスのfsckと、更新中の読み書きログを確認します。読み書きの観測はサンプルなので、結果と一緒に最大所要時間を確認します。

途中のセルが失敗した場合は、その場で操作を止め、サービス状態、`haadmin`、`rollingUpgrade query`、観測ログを確認します。更新セルの機械的な再実行や、手動での強制Active切り替えは避けてください。観測だけ終了する場合は `upgrade.stop_probe()` を呼び出せます。公式手順のrollbackにはクラスタ停止が必要で、準備後の書き込みは復元対象外です。


In [ ]:
upgrade.verify()

## 9. 確定

観測結果を確認し、検証クラスタで更新を確定する場合だけ下の値を変更してください。確定するとHDFSのローリングアップグレードのロールバック情報が破棄されます。


In [ ]:
finalize_confirmed = False
if finalize_confirmed:
    upgrade.finalize()


In [ ]:
import json
print(json.dumps(upgrade.evidence(), ensure_ascii=False))